# Phase 2 - Patients Clustering

**Course:** SWE485 (Selected Topics in Software Engineering)
**Phase:** 2 (Unsupervised Learning)

# The Notebook Overview




## 1. Clustering Rationale & Algorithm Selection
In this section, we will justify the choice of each .....?...... 

## 2. Data Preparation 

### 2.1 Feature scaling
Feature scaling is important in clustering because most clustering algorithms rely on distance calculations to measure similarity between data points. When variables are measured on different scales, features with larger numerical ranges can dominate the distance computation, leading to biased clustering results. For example, a variable such as RestingBP may have larger values compared to Oldpeak, causing it to have a disproportionate influence on cluster formation if not scaled.

Applying feature scaling ensures that all numerical variables contribute equally to the clustering process, allowing clusters to reflect meaningful patterns in the data rather than differences in measurement scale.
In this study, feature scaling was applied during preprocessing to continuous variables, including Age, MaxHR, Oldpeak, and RestingBP.

### 2.2 target removal

Target removal is necessary in clustering because clustering is an unsupervised learning task. This means the model should discover patterns and group similar data points without using any predefined labels.
Including the target variable may bias the clustering results, as it would indirectly guide the model toward known outcomes instead of uncovering natural structures in the data.

### Import Libraries

In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from kmodes.kprototypes import KPrototypes

### Load Preprocessed Dataset & remove target

In [ ]:
# Load the preprocessed dataset
# This dataset is already cleaned, encoded, and scaled from the EDA phase
# Shape: 917 rows × 21 columns (20 features + 1 target)
DATA_PATH = "Dataset/preprocessed_heart_data.csv"

original_data = pd.read_csv(DATA_PATH)

print(f"Dataset with the target label: {original_data.shape[0]} rows x {original_data.shape[1]} columns")

# Remove the target column for clustering (unsupervised learning)
clustering_data = original_data.drop(columns=["HeartDisease"])

print(f"Clustering data shape: {clustering_data.shape}")

clustering_data.head()

Dataset with the target label: 917 rows x 21 columns
Clustering data shape: (917, 20)


,Age,Sex,RestingBP,FastingBS,MaxHR,ExerciseAngina,Oldpeak,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up,Chol_category_Desirable,Chol_category_Borderline High,Chol_category_High
0,-1.432206,1,0.5,0,1.383339,0,-0.400000,0,1,0,0,0,1,0,0,0,1,0,0,1
1,-0.478057,0,1.5,0,0.754736,0,0.266667,0,0,1,0,0,1,0,0,1,0,1,0,0
2,-1.750256,1,0.0,0,-1.523953,0,-0.400000,0,1,0,0,0,0,1,0,0,1,0,0,1
3,-0.584074,0,0.4,0,-1.131075,1,0.600000,1,0,0,0,0,1,0,0,1,0,0,1,0
4,0.052026,1,1.0,0,-0.581047,0,-0.400000,0,0,1,0,0,1,0,0,0,1,1,0,0



## 3. Cluster Determination & Implementation

- Determine the optimal number of clusters.
- Apply clustering and assign cluster labels.

## 4. Evaluation Metrics & Visualizations
- Evaluate cluster quality using:
  - Silhouette Score
  - Davies-Bouldin Index
  - Within-Cluster Sum of Squares (WCSS)
  - BCubed Precision/Recall (optional, for external validation if ground truth exists)
- Visualize clusters:
  - PCA/t-SNE for 2D/3D projection of clusters
  - Feature importance per cluster
  - Cluster size distribution

## 5. Cluster Interpretation & Profiles
  - Profile each cluster: What characterizes each group?
  - Relate clusters to your domain: What do these groups mean for your advice system?

## 6. Integration Strategy (how clusters will enhance the system)
  - Integration proposal. If integration is not feasible, provide a detailed, justified explanation.

## 5. Challenges & Limitations

---

# **K-Prototypes**

## Section 1. Clustering Rationale & Algorithm Selection

### **What is K-Prototypes Algorithm?**
K-Prototypes is a clustering algorithm designed to handle mixed-type data, meaning datasets that contain both numerical and categorical features. It was introduced by Zhexue Huang (1997) as an extension of K-Means and K-Modes. While K-Means algorithm operates only on numerical data and K-Modes algorithm handles categorical data, many real-world datasets (such as medical data) contain a combination of both. K-Prototypes addresses this limitation by integrating the two approaches into a unified framework.


### **How does it work?**
K-Prototypes groups data points by measuring how similar they are to cluster centers (called prototypes). Unlike K-Means, it uses two different distance measures depending on the feature type:
- numerical features → Euclidean distance
- categorical features → simple matching (same or different)

The dissimilarity formula of K-Prototypes combines numerical and categorical components into a single measure used to assign data points to clusters, as defined by the paper (Huang, 1997):


| **Dissimilarity Formula** | $$ d(x, p) = s_n(x, p) + \gamma \, s_c(x, p) $$ | Combined distance for numerical and categorical features |
|---------------------------|-----------------------------------------------|----------------------------------------------------------|
| **Component** | **Formula** | **Description** |
| Numerical : $$s_n(x, p) $$ | $$ \sum_{j \in Num} (x_j - p_j)^2 $$ | Measures the squared euclidean distance between numerical featuresو, capturing how far a data point is from the cluster prototype in the numerical space.|
| Categorical : $$s_c(x, p)$$ | $$ \sum_{j \in Cat} \delta(x_j, p_j) $$ | Counts mismatches between categorical features, reflecting how many attributes differ between the data point and the prototype. |
| Categorical Matching function : $$ \delta(x_j, p_j)$$ | $$ \begin{cases} 0 & \text{if } x_j = p_j \\ 1 & \text{if } x_j \ne p_j \end{cases} $$ | Returns 0 if values match and 1 otherwise |
| Weight (gamma) : $$\gamma$$ | — | Controls the balance between numerical and categorical components. A larger value gives more importance to categorical differences, while a smaller value emphasizes numerical distance. |

### **Why is K-Prototypes suitable for our data?**

#### i. Dataset size & feature types

The dataset contains 917 samples and 20 features, combining both numerical and categorical variables. Numerical features (such as age, resting blood pressure, maximum heart rate, and oldpeak) were scaled to ensure balanced contribution in distance calculations, while categorical variables include binary features (e.g., sex, exercise-induced angina) and nominal clinical attributes such as chest pain type, ECG results, ST slope, and cholesterol category.

This mixed structure directly matches the design of K-Prototypes, which handles numerical and categorical features using different dissimilarity measures. Numerical variables are compared using squared Euclidean distance, while categorical variables are compared using simple matching. This allows both continuous physiological measurements and discrete medical conditions to be incorporated without distortion or loss of meaning.

A key consideration is the presence of outliers in some numerical features, particularly resting blood pressure and oldpeak. As a centroid-based method, K-Prototypes is sensitive to such values since cluster centers are computed using the mean (Han et al., 2011). However, this effect is reduced through feature scaling and by the inclusion of categorical similarity, which helps stabilize clustering and ensures that clusters reflect consistent patient patterns rather than isolated extreme values.

In contrast, alternative methods do not preserve this balance. K-Means ignores categorical information, while K-Modes ignores numerical variation. Therefore, given the dataset’s mixed feature types, moderate size, and presence of outliers, K-Prototypes provides a suitable and well-justified approach.

---
#### ii. Expected cluster shapes

Cluster shape expectations depend on both the algorithm and the nature of the data. In general, clustering methods may identify spherical (compact), elongated, or arbitrary-shaped clusters depending on how similarity is defined (Bishop, 2006).

K-Prototypes, as a centroid-based algorithm, assumes clusters that are compact and approximately spherical in the numerical feature space, since it minimizes distance to a central prototype. Therefore, it is less suitable for detecting highly irregular or non-convex cluster structures.

However, this assumption is reasonable for the current dataset. The data represents structured clinical attributes rather than spatial or image-based patterns. Clusters are therefore expected to form based on similarity in patient profiles, rather than geometric shapes.

Patients with similar combinations of features (e.g., age, ECG results, cholesterol levels) naturally group together, resulting in compact and well-defined clusters. This aligns closely with the assumptions of K-Prototypes.


---

#### iii. Scalability & interpretability

- **Scalability** refers to the ability of a clustering algorithm to efficiently handle increasing data size and dimensionality. K-Prototypes demonstrates strong scalability, as it follows an iterative structure similar to K-Means, where each iteration assigns points to the nearest prototype and updates cluster representatives.

The computational cost is approximately proportional to:

$$
\mathcal{O}(n \cdot k \cdot d)
$$

This complexity is consistent with centroid-based clustering methods (Han et al., 2011).



- **Interpretability** refers to how easily the resulting clusters can be understood and explained. K-Prototypes provides interpretable cluster representations through its prototype definition:

$$
p_j =
\begin{cases}
\text{mean}(x_j), & j \in Num \\
\text{mode}(x_j), & j \in Cat
\end{cases}
$$

This means each cluster is described by:
- the **average** of numerical features  
- the **most frequent category** for categorical features  

As a result, clusters can be interpreted as meaningful patient profiles, combining both physiological measurements and clinical attributes. This level of transparency is particularly important in healthcare applications, where understanding cluster characteristics is essential for analysis and decision-making.



----
> Huang, Z. (1997). *Clustering Large Data Sets with Mixed Numeric and Categorical Values*.  
> https://link.springer.com/chapter/10.1007/3-540-63438-4_48
>
> Han, J., Kamber, M., & Pei, J. (2011). *Data Mining: Concepts and Techniques*.  
> https://www.sciencedirect.com/book/9780123814791/data-mining-concepts-and-techniques
>
> Bishop, C. M. (2006). *Pattern Recognition and Machine Learning*. Springer.  
> https://link.springer.com/book/10.1007/978-0-387-45528-0

## Section 3. Cluster Determination & Implementation